# TB Portals — Frozen Backbone Comparison
Runs R1 (BMC head) on three alternative frozen CXR backbones to validate that RAD-DINO is the right choice:
1. **BioMedCLIP** (vision-language, biomedical literature)
2. **TorchXRayVision** (CXR-specific, supervised on 14 datasets)
3. **DINOv2-natural** (Facebook DINOv2 on natural images)

Same head architecture, same protocol, same 5,010-image manifest, 5 seeds, 4 modes (a1/a2/a3/fusion).
Attach only `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Estimated runtime ≈ 3 hr.

## 0 — Clone repo

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH   = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
print('repo ready at', REPO_DIR)

## 1 — Install deps (open_clip for BioMedCLIP)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'open_clip_torch', 'torchxrayvision', 'pydicom',
                'pylibjpeg', 'pylibjpeg-libjpeg'], check=False)
print('deps installed')

## 2 — Paths and manifest

In [ ]:
import os
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
print('export exists:', os.path.isdir(KAGGLE_EXPORT))

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f'manifest: {len(paper_df)} images -> {PAPER_MANIFEST}')

## 3 — Cache features for each backbone (one CLS file per backbone)

In [ ]:
from cache_features import main as cache_main
BACKBONES = [
    ('biomedclip',     f'{WORK}/features_biomedclip_cls.npz'),
    ('txrv',           f'{WORK}/features_txrv_cls.npz'),
    ('dinov2-natural', f'{WORK}/features_dinov2nat_cls.npz'),
]
import os
for name, path in BACKBONES:
    if os.path.isfile(path):
        print('cached ->', name, '|', path); continue
    print(f'caching {name} ...')
    cache_main(['--manifest', PAPER_MANIFEST, '--out', path,
                '--backbone', name, '--batch-size', '32'])
print('all backbones cached')

## 4 — Train R1 (BMC) per backbone × mode × country × seed

In [ ]:
from src.training.train_agentic import main as agentic_main
import os
for name, path in BACKBONES:
    for mode in ['a2', 'a3', 'fusion', 'a1']:
        outdir = f'{WORK}/baseline_{name}_{mode}'
        if os.path.isdir(outdir):
            print('skip', outdir); continue
        print(f'=== {name} | {mode} ===')
        args = ['--features', path, '--manifest', PAPER_MANIFEST,
                '--mode', mode, '--out-dir', outdir,
                '--rungs', '1', '--seeds', '0', '1', '2', '3', '4',
                '--held-outs', 'Romania', 'Moldova', 'Kazakhstan']
        # a1 needs a patch-grid cache; skip a1 for non-RAD-DINO backbones for now
        if mode == 'a1' and name != 'rad-dino':
            print('  (a1 mode needs grid cache; skipping for backbones comparison run)')
            continue
        agentic_main(args)

## 5 — Compute pool-mean and kNN-only trivial baselines (CPU only)

In [ ]:
import numpy as np, pandas as pd, json
from src.data.tbportals import make_country_split
from cache_features import load_features

RAD = f'{WORK}/features_rad-dino_cls.npz'  # used by kNN baseline
if not os.path.isfile(RAD):
    print('caching RAD-DINO for kNN baseline...')
    cache_main(['--manifest', PAPER_MANIFEST, '--out', RAD,
                '--backbone', 'rad-dino', '--batch-size', '32'])
feats, dim = load_features(RAD)

rows_mean, rows_knn = [], []
manifest = pd.read_csv(PAPER_MANIFEST, dtype={'image_id': str})
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    tr_df, _, te_df = make_country_split(manifest, held_out_country=country, val_fraction=0.2, seed=0)
    tr_df = tr_df.copy(); te_df = te_df.copy()
    tr_df['timika'] = tr_df['alp_0_100'] + 40 * tr_df['cavity']
    te_df['timika'] = te_df['alp_0_100'] + 40 * te_df['cavity']

    # ---- pool-mean baseline ----
    yhat = float(tr_df['timika'].mean())
    mae_mean = float(np.mean(np.abs(te_df['timika'].values - yhat)))
    rows_mean.append({'method': 'pool_mean', 'held_out': country, 'n_test': len(te_df), 'timika_mae': mae_mean})

    # ---- kNN-only baseline (K=25) ----
    X_tr = np.stack([feats[i] for i in tr_df['image_id'] if i in feats])
    y_tr = tr_df[tr_df['image_id'].isin(feats)]['timika'].values
    X_te = np.stack([feats[i] for i in te_df['image_id'] if i in feats])
    y_te = te_df[te_df['image_id'].isin(feats)]['timika'].values
    X_tr_n = X_tr / (np.linalg.norm(X_tr, axis=1, keepdims=True) + 1e-9)
    X_te_n = X_te / (np.linalg.norm(X_te, axis=1, keepdims=True) + 1e-9)
    sim = X_te_n @ X_tr_n.T
    idx = np.argsort(-sim, axis=1)[:, :25]
    yhat = y_tr[idx].mean(axis=1)
    mae_knn = float(np.mean(np.abs(y_te - yhat)))
    rows_knn.append({'method': 'knn_only', 'held_out': country, 'n_test': len(y_te), 'timika_mae': mae_knn})

    print(f'{country}: pool_mean={mae_mean:.2f} | knn_only={mae_knn:.2f}')

trivial = pd.DataFrame(rows_mean + rows_knn)
trivial.to_csv(f'{WORK}/trivial_baselines.csv', index=False)
print('saved trivial baselines')

## 6 — Package results for download

In [ ]:
import shutil, glob, os
import pandas as pd
# collect every results_agentic_*.csv across backbone runs
rows = []
for name, _ in BACKBONES:
    for mode in ['a2', 'a3', 'fusion', 'a1']:
        for f in glob.glob(f'{WORK}/baseline_{name}_{mode}/results_agentic_*.csv'):
            d = pd.read_csv(f); d['backbone'] = name; d['mode'] = mode
            rows.append(d)
if rows:
    pd.concat(rows, ignore_index=True).to_csv(f'{WORK}/results_backbones.csv', index=False)
    print('aggregated results -> results_backbones.csv')

OUT = f'{WORK}/backbones_results'
os.makedirs(OUT, exist_ok=True)
for src in [f'{WORK}/results_backbones.csv', f'{WORK}/trivial_baselines.csv']:
    if os.path.isfile(src):
        shutil.copy(src, OUT)
zip_path = shutil.make_archive(f'{WORK}/backbones_results', 'zip', OUT)
print('zip ->', zip_path)